# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

This stage tracks two distinct wallet groups on the full trade stream (BUY + SELL):

- `copyable_group`: wallets that pass base eligibility and strong copyability thresholds.
- `predicting_group`: the remaining eligible wallets (not in copyable) that pass a predictability score threshold.

Notes:
- Group membership is threshold-based (absolute scores), not target-count based.
- Open-buy skill metrics are kept for diagnostics/legacy comparison.
- `selected_wallets_v2.parquet` stores both groups with a `wallet_group` label.
- Stage2 backtest uses only `predicting_group`.


In [2]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

## Configuration

In [3]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [4]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-05-31  TRAIN_A_END_DATE=2026-03-21


## Compute / load wallet skill metrics (legacy diagnostics)

These open-buy metrics are not used for the final copy-trigger wallet selection.


In [5]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,10964
train_b_wallets,10964
full_train_wallets,10964
test_wallets,10964


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [6]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,1813,0.07,0.12,1.52,103284.02,0.42
11,avg_copy_roi_capped,100,100,2695,0.07,0.10,1.12,143225.93,0.42
0,prob_edge_shrunk,50,50,4204,0.01,0.12,0.93,115130.45,0.47
22,roi_sharpe,200,200,3250,0.01,0.07,0.85,101101.91,0.70
12,avg_copy_roi_capped,200,200,5272,0.06,0.11,0.82,248959.70,0.48
1,prob_edge_shrunk,100,100,5418,0.03,0.11,0.78,201409.43,0.51
13,avg_copy_roi_capped,300,300,7046,0.06,0.07,0.76,384078.84,0.50
23,roi_sharpe,300,300,4467,0.01,0.07,0.70,117444.18,0.70
6,weighted_prob_edge_shrunk,100,100,2399,0.06,0.09,0.68,256835.63,0.47
21,roi_sharpe,100,100,2473,0.01,0.03,0.66,73063.49,0.78


In [7]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Full-stream wallet metrics for copy-trigger selection

Build per-wallet metrics on the full training trade stream (BUY + SELL).
Copyability is considered here (stage1), not in stage0 filtering.


In [8]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
].copy().reset_index(drop=True)

# Load full processed trades for the full-stream path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)

# df_full = df_full[
#     (df_full['avg_price'] <= 1) 
#     # (df_full['side'] == 'BUY') 
#     ()
#     ].copy().reset_index(drop=True)

df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"] = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"] = df_full["quantity"].astype(float)

# Full-stream PnL / notional across BUY + SELL
df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)

df_slice = df_full[df_full["is_train"]].copy()
wallet_vol_train, _ = compute_wallet_metrics(df_slice)
print(len(wallet_vol_train))


42222


In [9]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(
    wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'].replace(0, np.nan),
    0,
    1.0,
).fillna(0.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']

# Keep sorting deterministic for downstream filtering/inspection.
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)


In [10]:
wallet_cohorts = {}

In [11]:
print(len(wallet_vol_train))
wallet_vol_train.head()


42222


,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0xa231231e1e0a4d69fd7e51c3c6e6676b3f692e37,13.70,14,4,17,514.16,124.69,321.45,1.03,1.00,...,165.67,2025-12-22 15:47:30+00:00,118.12,307.14,2.46,0.20,0.00,0.24,1.00,118.12
1,0x3c6385912d528e4a71a5fb47ef269c960ec7a0df,5.17,2,1,20,40.14,376.45,391.36,1.00,1.00,...,99.00,2026-05-24 11:30:00+00:00,99.00,38.05,0.10,23.15,0.06,9.38,1.00,99.00
2,0x4b1d2d479f67a33141d67005acf069e10a08bdfb,NaN,1,1,1,1.52,150.48,150.48,1.00,1.00,...,99.00,2026-02-27 06:10:00+00:00,99.00,0.00,0.00,0.00,0.00,99.00,1.00,99.00
3,0x8da0d29ec66508950ad72846ac3164db7cc41c53,NaN,1,1,1,2.50,205.83,205.83,1.00,1.00,...,82.33,2026-04-03 19:40:00+00:00,82.33,0.00,0.00,0.00,0.00,82.33,1.00,82.33
4,0xdff348c95aa09169740de5a2d4ac8019bb3f531d,3.74,2,1,2,101.57,94.50,94.50,1.00,1.00,...,61.50,2026-05-14 22:35:00+00:00,61.50,100.00,1.06,100.00,1.06,0.93,1.00,61.50


In [12]:
# Final wallet selection: threshold-based copyable + predicting groups
# COPYABLE_SCORE_MIN = -1
# PREDICTING_SCORE_MIN = -2

candidates = wallet_vol_train.copy()

for col, default in {
    'top10_pnl_pct': np.nan,
    'top_market_abs_pnl_pct': np.nan,
    'market_pnl_hhi': np.nan,
    'positive_bucket_share': np.nan,
}.items():
    if col not in candidates.columns:
        candidates[col] = default

for col in [
    'average_roi', 'median_roi', 'num_buckets', 'num_markets',
    'pnl_volatility', 'max_drawdown_to_pnl',
    'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct', 'market_pnl_hhi',
    'copyable_roi', 'copyable_pnl_factor', 'copyable_pnl', 'total_notional',
    'max_copyable_drawdown_to_copyable_pnl', 'worst5_pnl_pct', 'positive_bucket_share',
]:
    if col in candidates.columns:
        candidates[col] = pd.to_numeric(candidates[col], errors='coerce')

base_mask = (
    (candidates['average_roi'] >= 0.1)
    # & (candidates['median_roi'] >= 0)
    & (candidates['num_buckets'] >= 20)
    & (candidates['num_markets'] >= 15)
    & (candidates['max_drawdown_to_pnl'] <= 0.2)
    & (candidates['top_market_pnl_pct'] < 0.25)
    & (candidates['top5_pnl_pct'] < 0.55)
    & (candidates['top10_pnl_pct'].fillna(0.85) < 0.8)
    & (candidates['top_market_abs_pnl_pct'].fillna(candidates['top_market_pnl_pct']) < 0.25)
    & (candidates['market_pnl_hhi'].fillna(0.20) < 0.20)
    & (candidates['median_dt'].dt.date <= (pd.Timestamp.today().date() - pd.Timedelta(days=30)))
    & (candidates['total_notional'] >= 5_000)
)

base_mask2 = (
    # (candidates['average_roi'] >= 0.1)
    # & (candidates['median_roi'] >= 0)
    (candidates['num_buckets'] >= 20)
    & (candidates['num_markets'] >= 15)
    & (candidates['max_drawdown_to_pnl'] <= 0.3)
    # & (candidates['top_market_pnl_pct'] < 0.3)
    & (candidates['top5_pnl_pct'] < 0.8)
    # & (candidates['top10_pnl_pct'].fillna(0.85) < 0.8)
    #  & (candidates['top_market_abs_pnl_pct'].fillna(candidates['top_market_pnl_pct']) < 0.5)
    # & (candidates['market_pnl_hhi'].fillna(0.20) < 0.20)
    & (candidates['median_dt'].dt.date <= (pd.Timestamp.today().date() - pd.Timedelta(days=30)))
    & (candidates['total_notional'] >= 1_000)
)

eligible_base = candidates[base_mask].copy()
if eligible_base.empty:
    raise ValueError('No wallets passed base eligibility filters.')

eligible_base['sample_mult'] = np.clip(
    np.log1p(eligible_base['num_buckets']) / np.log(2000.0),
    0.20,
    1.00,
)
eligible_base['downside_tail'] = eligible_base['worst5_pnl_pct'].abs().fillna(0.0)
eligible_base['predictability_score'] = eligible_base['sample_mult'] * (
    # 2.5 * eligible_base['average_roi'].fillna(0.0)
    # + 1.5 * eligible_base['median_roi'].fillna(0.0)
    + 1.2 * (eligible_base['positive_bucket_share'].fillna(0.5) - 0.5)
    - 1.0 * eligible_base['pnl_volatility'].fillna(0.0)
    - 1.2 * eligible_base['max_drawdown_to_pnl'].fillna(0.0)
    - 0.8 * eligible_base['top_market_abs_pnl_pct'].fillna(eligible_base['top_market_pnl_pct']).fillna(0.0)
    - 0.6 * eligible_base['market_pnl_hhi'].fillna(0.0)
    - 0.5 * eligible_base['downside_tail']
)
eligible_base['predicting_final_score'] = eligible_base['predictability_score']

copyable_mask = (
    (eligible_base['copyable_pnl'] > 0)
    & (eligible_base['average_roi'] >= 0.04)
    # & (eligible_base['copyable_pnl_factor'] >= 0.1)
    # & (eligible_base['copyable_roi'] >= 0.01)
    # & (eligible_base['median_roi'] >= 0.00)
    # & (eligible_base['average_roi'] >= 0.04)
    # & (
    #     eligible_base['max_copyable_drawdown_to_copyable_pnl']
    #     .fillna(eligible_base['max_drawdown_to_pnl'])
    #     .fillna(0.0)
    #     <= 0.3
    # )
)
copyable_candidates = eligible_base[copyable_mask].copy()
copyable_candidates['copyable_efficiency'] = copyable_candidates['copyable_pnl'].fillna(0.0) / (copyable_candidates['total_notional'].fillna(0.0) + 1.0)
copyable_candidates['copyable_dd_ratio'] = (
    copyable_candidates['max_copyable_drawdown_to_copyable_pnl']
    .fillna(copyable_candidates['max_drawdown_to_pnl'])
    .fillna(0.0)
)
copyable_candidates['copyable_score'] = (
    1.8 * copyable_candidates['copyable_roi'].fillna(0.0)
    + 1.2 * copyable_candidates['copyable_pnl_factor'].fillna(0.0)
    + 25.0 * copyable_candidates['copyable_efficiency'].clip(lower=-1.0, upper=1.0)
    - 0.8 * copyable_candidates['copyable_dd_ratio'].clip(lower=0.0)
    - 0.5 * copyable_candidates['top_market_abs_pnl_pct'].fillna(copyable_candidates['top_market_pnl_pct']).fillna(0.0)
)
copyable_candidates['final_score'] = (
    0.60 * copyable_candidates['predictability_score']
    + 0.40 * copyable_candidates['copyable_score']
)

wallet_cohorts['copyable_group'] = (
    copyable_candidates
    .sort_values('final_score', ascending=False)
    .reset_index(drop=True)
)

predicting_pool = eligible_base[
    ~eligible_base['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])
].copy()
wallet_cohorts['predicting_group'] = (
    predicting_pool
    # predicting_pool[predicting_pool['predicting_final_score'] >= PREDICTING_SCORE_MIN]
    .sort_values('predicting_final_score', ascending=False)
    .reset_index(drop=True)
)

if wallet_cohorts['copyable_group'].empty:
    raise ValueError('No wallets passed copyable-group thresholds.')
if wallet_cohorts['predicting_group'].empty:
    raise ValueError('No wallets passed predicting-group threshold.')

overlap = set(wallet_cohorts['copyable_group']['wallet']) & set(wallet_cohorts['predicting_group']['wallet'])
if overlap:
    raise ValueError(f'Wallet groups must be disjoint; found overlap of {len(overlap)} wallets.')

wallet_cohorts['copyable_group']['wallet_quality'] = wallet_cohorts['copyable_group']['final_score']
wallet_cohorts['predicting_group']['wallet_quality'] = wallet_cohorts['predicting_group']['predicting_final_score']

wallet_cohorts['multi_filter'] = wallet_cohorts['copyable_group'].copy()

print(f"Base-eligible wallets: {len(eligible_base):,}")
print(f"Copyable candidates: {len(copyable_candidates):,}")
print(f"Selected wallets (copyable group): {len(wallet_cohorts['copyable_group']):,}")
print(f"Selected wallets (predicting group): {len(wallet_cohorts['predicting_group']):,}")
wallet_cohorts['copyable_group'].head(10)



Base-eligible wallets: 156
Copyable candidates: 105
Selected wallets (copyable group): 105
Selected wallets (predicting group): 51


,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
0,0x6011655c4afb76f36dd1b08a137a1ba73466b31e,0.79,508,188,826,35342.02,4869.92,2161.07,0.22,0.33,...,2.36,0.82,0.01,-0.57,-0.57,0.06,0.02,6.28,2.17,2.17
1,0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,0.83,69,38,80,8009.87,1145.21,687.19,0.49,0.71,...,0.44,0.56,0.24,-0.50,-0.50,0.09,0.11,3.52,1.11,1.11
2,0x8eea2d8458fcad16bf50dec2b7de76e989d6e285,0.75,404,153,758,5477.14,1331.87,603.85,0.45,0.72,...,0.44,0.79,0.19,-0.92,-0.92,0.11,0.44,3.71,0.93,0.93
3,0x547b80b33e8dea565fd8a7cba3aa9aef85a4d0ba,0.67,651,227,791,11613.76,1158.72,965.76,0.43,0.67,...,0.12,0.85,0.22,-0.81,-0.81,0.08,0.08,3.18,0.79,0.79
4,0xb33e4e06175eab4561c5eca704b35cd93f5328c3,0.25,269,222,431,39180.27,719.46,305.12,0.47,0.70,...,0.49,0.74,0.03,0.11,0.11,0.01,0.02,1.51,0.67,0.67
5,0x5f612351a9e46afbe3f164473a7d77a69ec9840a,0.23,530,271,701,19992.89,656.10,337.64,0.23,0.40,...,0.34,0.83,0.26,-0.49,-0.49,0.02,0.15,1.51,0.31,0.31
6,0xbcc717d9fa42f3da102cc99c371504aa1aa1dda1,0.16,417,325,668,73004.70,809.31,273.44,0.38,0.58,...,0.04,0.79,0.09,0.14,0.14,0.00,0.17,0.38,0.24,0.24
7,0x919698b19427cbe6945b0dc823f2d9e126a4d934,0.31,1456,428,2235,38060.98,4052.73,1331.85,0.28,0.43,...,0.15,0.96,0.09,-0.59,-0.59,0.03,0.10,1.42,0.22,0.22
8,0x6d16c33b540069f5aa0d9f705b0d759dec927228,3.03,247,47,334,23668.18,8497.28,3276.21,0.40,0.65,...,0.30,0.73,0.25,-2.46,-2.46,0.14,0.25,4.19,0.20,0.20
9,0x8d0930676d559cc8fb7d8af0c555791c1820143f,0.24,3298,696,5902,1154417.40,11761.40,2730.44,0.26,0.40,...,0.11,1.00,0.10,-0.01,-0.01,0.00,0.12,0.40,0.16,0.16


In [13]:
copyable_candidates.sort_values('trade_count', ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score
15169,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0.13,15768,1750,22990,168896.62,2797.29,10.80,0.22,0.32,...,0.00,0.00,1.00,0.09,-0.31,-0.31,0.00,23.81,-19.06,-7.81
10443,0xa0bca9bdd8540da95060ed1fafb78aa03835d428,1.67,14997,1312,22292,1880952.25,63992.07,18820.06,0.32,0.47,...,0.29,0.05,1.00,0.25,-1.96,-1.96,0.01,0.41,0.35,-1.04
10918,0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,4.16,9974,330,16936,4025765.20,126675.59,26261.46,0.35,0.60,...,0.21,0.04,1.00,0.42,-4.67,-4.67,0.01,1.09,-0.41,-2.97
11058,0x22dbd51e1a4e10fff072fa24300238c64033190f,0.75,12097,2788,15742,936143.92,49830.31,15861.51,0.22,0.36,...,0.32,0.04,1.00,0.15,-0.76,-0.76,0.02,0.21,0.69,-0.18
11096,0xecc8b1562f4a521d0856e3f89c5bfa8bba969335,0.19,11030,2993,14159,134131.22,1068.49,207.85,0.34,0.58,...,0.19,0.04,1.00,0.35,-0.55,-0.55,0.00,0.45,-0.03,-0.34
10544,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,12.62,6806,1249,11113,5717143.33,356454.78,105507.42,0.40,0.68,...,0.30,0.05,1.00,0.33,-12.93,-12.93,0.02,0.33,0.61,-7.51
9159,0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,2.44,6252,1170,11032,3501602.26,95291.81,23792.36,0.33,0.49,...,0.25,0.08,1.00,0.21,-2.65,-2.65,0.01,0.29,0.36,-1.45
11660,0x6beefb1c8d4b91b9b6d437840c241e74e6997e78,0.59,5069,967,9692,34858.02,2286.47,248.35,0.35,0.60,...,0.11,0.03,1.00,0.29,-1.37,-1.37,0.01,2.06,-1.32,-1.35
11996,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,1.22,6419,1869,8717,443944.65,67290.40,7542.64,0.13,0.20,...,0.11,0.02,1.00,0.10,-1.16,-1.16,0.02,0.53,0.17,-0.63
6022,0x134a63b764ac7b008356e8db1857db94e6b09e42,4.43,6152,2301,8664,3280544.40,195126.51,80033.67,0.40,0.61,...,0.41,0.24,1.00,0.17,-4.40,-4.40,0.02,0.10,1.40,-2.08


In [14]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [15]:
filter_wallets = set(wallet_cohorts['multi_filter']["wallet"])
df_test =  df_full[
    (df_full['dt'] > pd.to_datetime("2026-04-16", utc=True))
    # & (df_full['wallet'] == '0xbacd00c9080a82ded56f504ee8810af732b0ab35')
    & (df_full['wallet'].isin(filter_wallets))
    ].groupby(['question', 'condition_id', 'outcome']).agg(
    # ].groupby(['wallet']).agg(
        pnl_sum=('pnl', 'sum'),
        trade_count=('pnl', 'size'),
        wallet_count=('wallet', 'nunique'),
    ).reset_index().sort_values('pnl_sum', key=abs, ascending=False)

big_conditions = df_test.groupby('condition_id').agg(
    abs_pnl_sum=('pnl_sum', lambda x: abs(x).sum()),
    min_abs_pnl_pct=('pnl_sum', lambda x: abs(x).min() / abs(x).sum() if abs(x).sum() > 0 else 0)
).reset_index().sort_values('abs_pnl_sum', ascending=False)



print(df_test['pnl_sum'].sum())
df_test = df_test[df_test['condition_id'].isin(big_conditions['condition_id'])].sort_values('question', ascending=False).merge(big_conditions, on='condition_id', how='left')

print(df_test[df_test['min_abs_pnl_pct']<0.1]['pnl_sum'].sum()) 

df_test[df_test['min_abs_pnl_pct'] < 0.1].sort_values(['abs_pnl_sum'], ascending=False).head(20)

797658.8226459998
556681.352842


,question,condition_id,outcome,pnl_sum,trade_count,wallet_count,abs_pnl_sum,min_abs_pnl_pct
26984,"US x Iran permanent peace deal by June 15, 2026?",0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e...,No,-338.84,96,12,270626.84,0.00
26985,"US x Iran permanent peace deal by June 15, 2026?",0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e...,Yes,270288.00,746,20,270626.84,0.00
22012,"Will Trump say ""Iran"" during events with Xi Ji...",0xaa145cf5714455f546afe53037b8712df749ba96c048...,No,48581.15,476,19,49433.49,0.02
22011,"Will Trump say ""Iran"" during events with Xi Ji...",0xaa145cf5714455f546afe53037b8712df749ba96c048...,Yes,852.34,83,9,49433.49,0.02
27088,US announces new Iran agreement/ceasefire exte...,0x6a8cfe84d17693425f27831db5949d7511f3393d4624...,Yes,-35993.18,411,11,36764.58,0.02
27087,US announces new Iran agreement/ceasefire exte...,0x6a8cfe84d17693425f27831db5949d7511f3393d4624...,No,771.39,129,7,36764.58,0.02
26993,"US x Iran permanent peace deal by April 30, 2026?",0xceb6dfaa2cf5abc9d47ebc867b984a77151049442492...,Yes,912.45,121,9,36394.42,0.03
26992,"US x Iran permanent peace deal by April 30, 2026?",0xceb6dfaa2cf5abc9d47ebc867b984a77151049442492...,No,35481.97,375,9,36394.42,0.03
27113,US announces new Iran agreement/ceasefire exte...,0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198...,No,-80.41,87,11,32730.05,0.00
27114,US announces new Iran agreement/ceasefire exte...,0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198...,Yes,32649.64,285,15,32730.05,0.00


In [16]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False).head(10)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
99,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,12.62,6806,1249,11113,5717143.33,356454.78,105507.42,0.40,0.68,-0.33,0.10,0.04,0.01,0.60,0.05,2025-10-27 23:22:30+00:00,0.16,67467.21,0.19,35112.36,0.33,0.06,0.30,0.05,1.00,0.33,-12.93,-12.93,0.02,0.33,0.61,-7.51,-7.51
91,0x134a63b764ac7b008356e8db1857db94e6b09e42,4.43,6152,2301,8664,3280544.40,195126.51,80033.67,0.40,0.61,-0.17,0.22,0.10,0.02,0.77,0.01,2025-08-25 20:25:00+00:00,0.58,18659.84,0.10,8073.65,0.10,0.06,0.41,0.24,1.00,0.17,-4.40,-4.40,0.02,0.10,1.40,-2.08,-2.08
103,0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,14.67,4032,618,7465,3878273.37,344099.43,67430.64,0.51,0.79,-0.40,0.20,0.11,0.02,0.53,0.01,2025-11-24 18:30:00+00:00,0.11,53241.93,0.15,50759.68,0.75,0.09,0.20,0.02,1.00,0.40,-15.13,-15.13,0.02,0.75,0.05,-9.06,-9.06
97,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,10.58,992,290,1874,1183401.66,163317.51,46422.23,0.44,0.66,-0.26,0.09,0.07,0.03,0.61,0.05,2025-12-19 21:30:00+00:00,0.80,13660.97,0.08,10133.74,0.22,0.14,0.28,0.23,0.91,0.26,-9.76,-9.76,0.04,0.22,1.52,-5.25,-5.25
88,0x1c144e30f405a25f991cbd8baa15d40599090869,3.67,4248,1048,5728,618405.30,82414.42,27903.99,0.36,0.54,-0.15,0.10,0.04,0.01,0.60,0.12,2026-01-27 20:45:00+00:00,0.20,10118.78,0.12,4941.62,0.18,0.13,0.34,0.07,1.00,0.15,-3.81,-3.81,0.05,0.18,1.50,-1.69,-1.69
95,0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,4.16,9974,330,16936,4025765.20,126675.59,26261.46,0.35,0.60,-0.42,0.09,0.05,0.02,0.50,0.01,2026-01-04 23:42:30+00:00,0.19,25181.82,0.20,28637.08,1.09,0.03,0.21,0.04,1.00,0.42,-4.67,-4.67,0.01,1.09,-0.41,-2.97,-2.97
92,0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,3.79,3675,1012,5355,922794.84,83469.52,24625.89,0.31,0.51,-0.24,0.14,0.04,0.01,0.50,-1.00,2025-09-24 20:20:00+00:00,0.39,12404.99,0.15,8691.36,0.35,0.09,0.30,0.12,1.00,0.24,-4.12,-4.12,0.03,0.35,0.93,-2.10,-2.10
77,0x41583f2efc720b8e2682750fffb67f2806fece9f,2.35,5963,1050,8602,999733.58,74458.97,24257.16,0.25,0.40,-0.23,0.06,0.03,0.01,0.48,-1.00,2025-10-23 16:00:00+00:00,0.40,6445.82,0.09,8899.39,0.37,0.07,0.33,0.13,1.00,0.23,-2.62,-2.62,0.02,0.37,0.92,-1.20,-1.20
84,0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,2.44,6252,1170,11032,3501602.26,95291.81,23792.36,0.33,0.49,-0.21,0.10,0.06,0.01,0.56,0.01,2025-10-10 17:00:00+00:00,0.33,10193.16,0.11,6836.91,0.29,0.03,0.25,0.08,1.00,0.21,-2.65,-2.65,0.01,0.29,0.36,-1.45,-1.45
75,0x88b59d79b6e1659c95a0043028e5bb7a26e6205c,4.19,296,36,647,475683.39,59932.81,23172.65,0.44,0.67,-0.15,0.18,0.17,0.08,0.84,0.16,2025-06-23 19:57:30+00:00,0.19,2775.43,0.05,1690.00,0.07,0.13,0.39,0.08,0.75,0.15,-3.07,-3.07,0.05,0.07,1.67,-1.17,-1.17


In [17]:
df_slice.head(1)

,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional
0,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c...,2576518463476533936382780799241198667006764167...,2026-05-18 05:53:53+00:00,BUY,5000.00,5000.00,0.14,700.00,0.00,-700.00,-56.59,False,0.00,2026-05-22 06:11:42+00:00,0x0608e4eeb4f28612dfee57a202a790d2f183d4e251f0...,1,True,404.23,52.55,2.00,2026-05-21T00:00:00Z,Iran closes its airspace by May 21?,"[Politics, Iran, Geopolitics, U.S. x Iran]",Politics,1121356112270496321028332968926495085221299898...,Yes,-700.00,700.00


In [18]:
wallets = set(wallet_cohorts['copyable_group']['wallet'])

df = df_full #[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

33706813
102245


In [19]:
wallet_set = set(wallet_cohorts['copyable_group']['wallet'])
print(f"Copyable wallets: {len(wallet_set)} with fills {df_wallets[df_wallets['wallet'].isin(wallet_set)].shape[0]} with total PnL: {df_wallets[df_wallets['wallet'].isin(wallet_set)]['pnl'].sum():,.2f}")
df_wallets[df_wallets['wallet'].isin(wallet_set)]['trade_pnl'].sum()

Copyable wallets: 105 with fills 102245 with total PnL: 473,987.76


np.float64(473987.7562129999)

In [20]:
df_wallets[df_wallets['wallet'] == '0x969fae0a3a93778adc42178f72c612ed8c4e4d55'].groupby(['wallet', 'is_train']).agg({'trade_pnl': 'sum'}).sort_values(by='trade_pnl', ascending=True).head(20)

,,trade_pnl
wallet,is_train,


In [21]:
MARKET = '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty=('copyable_qty', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: condition_id       0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...
end_date_iso                                    2026-01-31T00:00:00Z
question                       US strikes Iran by February 28, 2026?
tags               [Politics, Iran, Middle East, Geopolitics, Wor...
primary_tag                                                 Politics
winner_token_id    1107900031214423651268558640767076860146505232...
outcome                                                          Yes
Name: 59766, dtype: object


In [22]:
# ── positions per wallet + sum, dual-axis (position + price) ──────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(dfc) != 0:

    # ── 1. build per-bucket price series for YES (volume-weighted avg)
    _yes = dfc[dfc['outcome'] == 'Yes'].copy()
    _yes['_wv'] = _yes['avg_price'] * _yes['quantity']
    price_ts = (
        _yes.groupby('dt')[['_wv', 'quantity']].sum()
        .rename(columns={'quantity': '_qty'})
        .assign(vwap=lambda d: d['_wv'] / d['_qty'])[['vwap']]
        .reset_index()
        .sort_values('dt')
    )

    # ── 2. net YES position per wallet per timestamp (YES pos - NO pos)
    # position is cumulative after each trade; take last value per (dt, wallet, outcome)
    pos_per_wallet = (
        dfc
        .groupby(['dt', 'wallet', 'outcome'])['position']
        .last()
        .reset_index()
    )
    pos_yes = pos_per_wallet[pos_per_wallet['outcome'] == 'Yes'].rename(columns={'position': 'pos_yes'})[['dt', 'wallet', 'pos_yes']]
    pos_no = pos_per_wallet[pos_per_wallet['outcome'] == 'No'].rename(columns={'position': 'pos_no'})[['dt', 'wallet', 'pos_no']]
    net_pos = (
        pos_yes
        .merge(pos_no, on=['dt', 'wallet'], how='outer')
        .fillna(0)
    )
    net_pos['net'] = net_pos['pos_yes'] - net_pos['pos_no']
    net_pos = net_pos.sort_values(['wallet', 'dt']).reset_index(drop=True)

    # ── 3. weight wallets by proper probabilistic score (Brier skill) with shrinkage
    wallet_scores = (
        train_b_metrics[['wallet', 'brier_skill', 'open_buy_trades']]
        .drop_duplicates('wallet')
        .copy()
    )
    wallet_scores['score'] = wallet_scores['brier_skill'].clip(lower=0.0).fillna(0.0)
    wallet_scores['confidence'] = (
        wallet_scores['open_buy_trades'].fillna(0.0)
        / (wallet_scores['open_buy_trades'].fillna(0.0) + 25.0)
    )
    wallet_scores['wallet_weight'] = wallet_scores['score'] * wallet_scores['confidence']
    wallet_weights = wallet_scores.set_index('wallet')['wallet_weight']

    # ── 4. normalize by each wallet's typical move size in this market
    net_pos['net_step'] = net_pos.groupby('wallet')['net'].diff()
    typical_move = net_pos.groupby('wallet')['net_step'].apply(
        lambda s: s.abs().dropna().median() if s.notna().any() else np.nan
    ).rename('typical_move')
    positive_moves = typical_move[typical_move > 0]
    fallback_move = positive_moves.median() if not positive_moves.empty else 1.0
    typical_move = typical_move.fillna(fallback_move).clip(lower=fallback_move * 0.25)

    # ── 5. carry positions forward so the signal reflects current wallet stance
    timeline = price_ts['dt'].sort_values().drop_duplicates()
    net_panel = (
        net_pos.pivot(index='dt', columns='wallet', values='net')
        .sort_index()
        .reindex(timeline)
        .ffill()
        .fillna(0.0)
    )
    available_wallets = net_panel.columns.intersection(wallet_weights.index).intersection(typical_move.index)
    net_panel = net_panel[available_wallets]
    wallet_weights = wallet_weights.reindex(available_wallets).fillna(0.0)
    typical_move = typical_move.reindex(available_wallets)
    normalized_panel = net_panel.divide(typical_move, axis=1)
    weighted_panel = normalized_panel.multiply(wallet_weights, axis=1)

    signal_ts = pd.DataFrame({
        'dt': net_panel.index,
        'total_net': net_panel.sum(axis=1).to_numpy(),
        'weighted_position': weighted_panel.sum(axis=1).to_numpy(),
    })

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=['YES token (positive weighted signal => predicts YES)'],
        specs=[[{'secondary_y': True}]],
    )

    COLORS = [
        '#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66',
        '#77BEDB', '#F7A35C', '#90ED7D', '#8085E9', '#F15C80',
    ]

    top_wallets = net_panel.abs().max().sort_values(ascending=False).head(10).index

    # ── per-wallet net position (step: hold latest value, no interpolation)
    for c_idx, wallet in enumerate(top_wallets):
        wdf = (
            net_panel[[wallet]].reset_index()
            .rename(columns={wallet: 'net'})
            .sort_values('dt')
        )
        fig.add_trace(
            go.Scatter(
                x=wdf['dt'],
                y=wdf['net'],
                mode='lines',
                name=f'{wallet[:8]}...',
                line=dict(color=COLORS[c_idx % len(COLORS)], width=1, shape='hv'),
                legendgroup=wallet,
                opacity=0.6,
            ),
            row=1, col=1, secondary_y=False,
        )

    # ── total net position line (primary y, bold)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['total_net'],
            mode='lines',
            name='SUM (net YES)',
            line=dict(color='#222222', width=3, shape='hv'),
            legendgroup='sum_net',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── weighted normalized wallet position (primary y)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['weighted_position'],
            mode='lines',
            name='Weighted normalized position',
            line=dict(color='#C44E52', width=3, shape='hv'),
            legendgroup='weighted_signal',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── YES price line (secondary y)
    fig.add_trace(
        go.Scatter(
            x=price_ts['dt'],
            y=price_ts['vwap'],
            mode='lines+markers',
            name='Price (YES)',
            line=dict(color='#888888', width=2, dash='dot', shape='hv'),
            marker=dict(size=4),
            legendgroup='price_yes',
        ),
        row=1, col=1, secondary_y=True,
    )

    fig.add_hline(y=0, line=dict(color='#BBBBBB', width=1, dash='dash'))
    fig.update_yaxes(title_text='Net position / weighted normalized total', row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text='Price (USDC)', row=1, col=1, secondary_y=True, range=[0, 1])

    fig.update_layout(
        title=f'Wallet net YES positions and weighted signal - {MARKET[:16]}...',
        height=500,
        template='plotly_white',
        legend=dict(orientation='v', x=1.05),
    )
    fig.show(renderer='browser')


In [23]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [24]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

105


,wallet,primary_tag,primary_tag_qty,primary_tag_ratio,pnl
765,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,Politics,10501268.60,0.82,390479.21
129,0x134a63b764ac7b008356e8db1857db94e6b09e42,Politics,5243484.96,0.81,382922.04
1106,0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,Politics,6673432.99,0.79,345102.32
915,0x777fae71d2ff9ec48a1213d48ba1d9d91024a1bb,Finance,2461825.70,0.81,156059.76
477,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,Politics,2065020.16,0.89,143508.71
1471,0xd3989ba133ab48b5b3a81e3dba9b37b5966a46d7,Politics,2242718.97,0.94,98098.82
1458,0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,Politics,6142868.19,0.69,92281.81
699,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,Politics,1418521.44,0.91,91827.70
1578,0xee50a31c3f5a7c77824b12a941a54388a2827ed6,Business,627189.40,0.46,89934.42
211,0x1c144e30f405a25f991cbd8baa15d40599090869,Politics,1208168.09,0.89,80717.94


In [25]:
# from polymarket_analysis.data.tags import load_tag_map
# from polymarket_analysis.data.tags import dominant_tag_stats

# split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

# _result = dominant_tag_stats(
#     df_trades=df_full[df_full['dt'] >= split_dt],
#     wallets=wallet_cohorts["multi_filter"]["wallet"],
# )

# print(f"Wallets: {len(_result)}")
# _result[_result['primary_tag'] == 'Politics'].head(5)


In [26]:
# _result.groupby('primary_tag').agg(
#     tag_pnl=('tag_pnl', 'sum'),
#     wallets=('wallet', 'nunique')
#     )

In [27]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [28]:
import plotly.graph_objects as go

print(f"copyable_group wallets: {len(wallet_cohorts['copyable_group']):,}")
print(f"predicting_group wallets: {len(wallet_cohorts['predicting_group']):,}")

pd.set_option('display.max_rows', 100)

copyable_view = wallet_cohorts['copyable_group'].sort_values('final_score', ascending=False).head(20)
predicting_view = wallet_cohorts['predicting_group'].sort_values('predicting_final_score', ascending=False).head(20)

copyable_view, predicting_view

plot_start = pd.Timestamp(END_DATE_TRAIN, tz='UTC') + pd.Timedelta(days=1)
plot_rows = []
for group_label, cohort_name in [('copyable', 'copyable_group'), ('predicting', 'predicting_group')]:
    wallets = set(wallet_cohorts[cohort_name]['wallet'])
    group_trades = df_full[
        (df_full['wallet'].isin(wallets))
        & (df_full['dt'] >= plot_start)
    ][['dt', 'pnl', 'copyable_pnl']].copy()

    daily = (
        group_trades
        .assign(plot_dt=group_trades['dt'].dt.floor('D'))
        .groupby('plot_dt', as_index=False)[['pnl', 'copyable_pnl']]
        .sum()
        .sort_values('plot_dt')
        .reset_index(drop=True)
    )
    daily['cum_trade_pnl'] = daily['pnl'].cumsum()
    daily['cum_copyable_pnl'] = daily['copyable_pnl'].cumsum()
    daily['wallet_group'] = group_label
    plot_rows.append(daily)

group_perf = pd.concat(plot_rows, ignore_index=True)

fig = go.Figure()
colors = {'copyable': '#1f77b4', 'predicting': '#ff7f0e'}
for group_label in ['copyable', 'predicting']:
    g = group_perf[group_perf['wallet_group'] == group_label]
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_trade_pnl'],
            mode='lines',
            name=f'{group_label} trade pnl',
            line=dict(color=colors[group_label], width=2),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_copyable_pnl'],
            mode='lines',
            name=f'{group_label} copyable pnl',
            line=dict(color=colors[group_label], width=2, dash='dash'),
        )
    )

fig.update_layout(
    title='Stage1 wallet-group performance on test period',
    xaxis_title='Date',
    yaxis_title='Cumulative PnL (USDC)',
    template='plotly_white',
)
fig.show(renderer='browser')


copyable_group wallets: 105
predicting_group wallets: 51


In [29]:
copyable_wallets = wallet_cohorts['copyable_group'].copy().assign(wallet_group='copyable')
predicting_wallets = wallet_cohorts['predicting_group'].copy().assign(wallet_group='predicting')

selected_wallets = (
    pd.concat([copyable_wallets, predicting_wallets], ignore_index=True)
    .drop_duplicates(subset=['wallet', 'wallet_group'])
    .reset_index(drop=True)
)

selected_wallets.to_parquet(WORKSPACE_DIR / 'selected_wallets_v2.parquet', index=False)
print(f"Saved selected wallets: {len(selected_wallets):,}")
print(selected_wallets['wallet_group'].value_counts().to_dict())


Saved selected wallets: 156
{'copyable': 105, 'predicting': 51}


In [30]:
WORKSPACE_DIR / "selected_wallets_v2.parquet"


PosixPath('../../data/trade_signals_workspace_v2/selected_wallets_v2.parquet')

## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [31]:
from polymarket_analysis.strategy.definition import TriggerSpec, WalletSelectionStrategy
from polymarket_analysis.strategy.registry import save_strategy

SIGNAL_THRESHOLD = 0.65

selected_wallets = wallet_cohorts['predicting_group'].copy().reset_index(drop=True)
predicting_wallets = selected_wallets.copy()
if 'wallet_quality' not in predicting_wallets.columns:
    if 'predicting_final_score' in predicting_wallets.columns:
        predicting_wallets['wallet_quality'] = predicting_wallets['predicting_final_score']
    else:
        predicting_wallets['wallet_quality'] = 0.0

strategy_specs = [
    (
        'predicting_group__score_threshold',
        f'predicting_group | score >= {SIGNAL_THRESHOLD:.2f} (Kelly)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.score_threshold',
            params={'threshold': SIGNAL_THRESHOLD, 'dynamic_sizing': True},
            mode='frame',
        ),
    ),
    (
        'predicting_group__all_open_buys',
        'predicting_group | all open-buys (fixed stake)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.all_open_buys',
            params={'dynamic_sizing': False},
            mode='frame',
        ),
    ),
    (
        'predicting_group__copy_triggers',
        'predicting_group | copy all triggers (tight slippage)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.copy_triggers',
            params={'dynamic_sizing': False},
            mode='frame',
        ),
    ),
]

all_strategies = []
for strategy_id, name, trigger in strategy_specs:
    all_strategies.append(
        WalletSelectionStrategy(
            strategy_id=strategy_id,
            name=name,
            selection_method='predicting_group',
            trigger=trigger,
            wallets=predicting_wallets.copy(),
            params={
                'selection_metric': 'predicting_final_score',
                'top_n': len(predicting_wallets),
                'signal_threshold': SIGNAL_THRESHOLD if trigger.fn_ref.endswith('score_threshold') else None,
            },
            metadata={
                'cohort': 'predicting_group',
                'wallet_group': 'predicting',
                'selection_metric': 'predicting_final_score',
                'signal_threshold': SIGNAL_THRESHOLD,
                'top_n': len(predicting_wallets),
                'end_date_train': str(END_DATE_TRAIN),
                'train_a_end_date': str(TRAIN_A_END_DATE),
                'selection_notes': (
                    'threshold-based: copyable group uses copyability quality gates, '
                    'predicting group uses predictability threshold on remaining wallets'
                ),
            },
        )
    )

for strategy in all_strategies:
    parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
    print(f"Saved [{strategy.strategy_id}] wallets={len(strategy.wallets):3d} trigger={strategy.trigger.fn_ref.split('.')[-1]}")

print(f'\nTotal strategies saved: {len(all_strategies)}')


Saved [predicting_group__score_threshold] wallets= 51 trigger=score_threshold
Saved [predicting_group__all_open_buys] wallets= 51 trigger=all_open_buys
Saved [predicting_group__copy_triggers] wallets= 51 trigger=copy_triggers

Total strategies saved: 3


## Strategy registry summary

In [32]:
from polymarket_analysis.strategy.registry import load_all_strategies

registry = load_all_strategies(WORKSPACE_DIR)
summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        'strategy_id': s.strategy_id,
        'name': s.name,
        'selection_method': s.selection_method,
        'num_wallets': len(s.wallets),
        'trigger_fn': s.trigger.fn_ref.split('.')[-1],
        'threshold': s.trigger.params.get('threshold'),
        'dynamic_sizing': s.trigger.params.get('dynamic_sizing'),
    })

pd.DataFrame(summary_rows).sort_values('strategy_id').reset_index(drop=True)


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__al...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | a...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,all_open_buys,NaN,False
1,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__co...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | c...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,copy_triggers,NaN,False
2,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__sc...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | s...,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,score_threshold,0.65,True
3,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__al...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | a...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,all_open_buys,NaN,False
4,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__co...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | c...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,copy_triggers,NaN,False
5,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__sc...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | s...,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,score_threshold,0.65,True
6,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__al...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | a...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,all_open_buys,NaN,False
7,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__co...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | c...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,copy_triggers,NaN,False
8,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__sc...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | s...,0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,score_threshold,0.65,True
9,0x64e86fefd50cdb6259eb00ff146d9aac03497819__al...,0x64e86fefd50cdb6259eb00ff146d9aac03497819 | a...,0x64e86fefd50cdb6259eb00ff146d9aac03497819,1,all_open_buys,NaN,False


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [33]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [34]:
df_fills = df_full.copy()
df_fills['copyable_pnl_exposure'] = np.where( 
    df_fills['copyable_qty'] > 0,
    df_fills['price'] * df_fills['copyable_qty'],
    np.nan
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

len(df_fills), len(df_fills[df_fills["dt"] < split_dt]), len(df_fills[df_fills["dt"] >= split_dt])

(33706813, 23865975, 9840838)

In [35]:
markets = df_fills[(df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))][['condition_id', 'tags', 'primary_tag']]
markets = markets.groupby('condition_id').agg(
    tags=('tags', 'first'),
    primary_tag=('primary_tag', 'first'),
).reset_index()
markets['has_mentions'] = markets['tags'].apply(lambda tags: 'Mentions' in tags)
mention_markets = set(markets[markets['has_mentions']]['condition_id'])
del markets
len(mention_markets)

9153

In [36]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    & (df_fills['primary_tag'] == 'Politics')
    & (~df_fills['condition_id'].isin(mention_markets))
    & (df_fills["dt"] >= split_dt)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
filter_fills = filter_fills[filter_fills['copyable_qty'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

10163
3927


,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,copyable_pnl_exposure
0,0x134a63b764ac7b008356e8db1857db94e6b09e42,0x0e2e5c0146c730a2e4ef8031c9128b4efef451a9f483...,2460603944479262807234872848621861946638682108...,2026-06-03 05:26:30+00:00,BUY,516.80,516.80,0.31,158.78,516.80,358.02,77.38,True,1.00,2026-06-09 00:24:34+00:00,0xbac40267ce348858289cf36ae0d400bcfcb666434cce...,3,False,111.70,170.12,5.00,2026-06-02T00:00:00Z,Will Karen Bass & Spencer Pratt advance to the...,"[Politics, Elections, US Election, Los Angeles...",Politics,2460603944479262807234872848621861946638682108...,No,358.02,158.78,34.32
1,0x134a63b764ac7b008356e8db1857db94e6b09e42,0x03d65c5eb801f03378ee2f23d55286d4ef7626f83125...,1543682689708755380186304440928712992082355291...,2026-06-05 17:52:12+00:00,BUY,2282.52,2282.52,0.99,2262.10,2282.52,20.42,2.20,True,1.00,2026-06-05 19:26:01+00:00,0x7f4591cf23203002a6c702bea03867a968f554292ca2...,11,False,246.24,10.98,11.00,2026-06-30T00:00:00Z,Will Donald Trump publicly insult someone on J...,"[Politics, Trump, Culture, Trump Daily]",Politics,1543682689708755380186304440928712992082355291...,Yes,20.42,2262.10,244.04


In [37]:
pd.set_option('display.max_colwidth', 100)

In [38]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor('5min')

MAX_EXPOSURE = 100

df = filter_fills.groupby(['bucket', 'condition_id', 'wallet', 'side']).agg(
    copyable_pnl_exposure=('copyable_pnl_exposure', 'sum'),
    total_copyable_qty=('copyable_qty', 'sum'),
    trade_pnl =('trade_pnl', 'sum'),
    copyable_pnl = ('copyable_pnl', 'sum'),
    trades=('condition_id', 'count'),
    copyable_qty=('copyable_qty', 'sum'),
    wallets=('wallet', lambda x: x.nunique()),
).reset_index()

scale = np.minimum(1, MAX_EXPOSURE / df["copyable_pnl_exposure"].replace(0, np.nan))

df = df.assign(
    scaled_copyable_pnl_exposure = df["copyable_pnl_exposure"] * scale,
    scaled_copyable_pnl = df["copyable_pnl"] * scale,
    scaled_copyable_qty = df["copyable_qty"] * scale,
)

df.head(10)

,bucket,condition_id,wallet,side,copyable_pnl_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,copyable_qty,wallets,scaled_copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_qty
0,2026-06-01 02:00:00+00:00,0xf5acdc8d50f9b6e98d5b9162a33919048ea2ccd08da175eb3c38f817e8a4bea9,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,BUY,63.44,72.09,12.00,8.65,1,72.09,1,63.44,8.65,72.09
1,2026-06-01 02:05:00+00:00,0x6a8cfe84d17693425f27831db5949d7511f3393d4624b182ac6956164cd32b10,0x0cb10c40b0776e9ee8cef970af85724654dda76c,BUY,0.02,8.20,-0.02,-0.02,1,8.20,1,0.02,-0.02,8.20
2,2026-06-01 03:05:00+00:00,0xd8664606498aba2a8ccab56656860cb3f02d74de80e974ab87a9f940e65d4d5e,0xe5b1377410ddd77dc3bf6fd5172e13fdc31330b2,BUY,16.56,18.00,1.44,1.44,1,18.00,1,16.56,1.44,18.00
3,2026-06-01 03:30:00+00:00,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,0x0cb10c40b0776e9ee8cef970af85724654dda76c,BUY,0.33,2.50,870.00,2.18,1,2.50,1,0.33,2.18,2.50
4,2026-06-01 03:35:00+00:00,0x366f89649caea042c96ee741b185461ec7faa408a2664ec44469a0061924b537,0x0cb10c40b0776e9ee8cef970af85724654dda76c,BUY,167.22,179.81,70.00,12.59,1,179.81,1,100.00,7.53,107.53
5,2026-06-01 03:40:00+00:00,0xf526b2fbff94ae996818e76c8b54cc99ffbfa0a1a9972a48253ada65e32786f7,0x0cb10c40b0776e9ee8cef970af85724654dda76c,BUY,4.63,51.46,-20.63,-4.63,1,51.46,1,4.63,-4.63,51.46
6,2026-06-01 04:20:00+00:00,0x328e31a593dab3ae872e451e4537a5846c6e3d00817a3dc1387f6c320d8d9889,0x83c56c56bc67c0b99b38abd1c0a3c1aca7a7e193,BUY,43.59,44.69,2.46,1.10,1,44.69,1,43.59,1.10,44.69
7,2026-06-01 05:00:00+00:00,0x92bbf16232980814da9a115206db8336feda24e91bda100a6ed126624effc1d9,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,BUY,6.00,10.82,213.52,4.82,1,10.82,1,6.00,4.82,10.82
8,2026-06-01 05:30:00+00:00,0xfd257d6247eb24d313d9b921a118c1754d03b3444f32a70334a28ef28d3e96b5,0x83c56c56bc67c0b99b38abd1c0a3c1aca7a7e193,BUY,18.71,18.88,0.17,0.17,1,18.88,1,18.71,0.17,18.88
9,2026-06-01 05:40:00+00:00,0xed25d03a2589af03a7a603e7c07c36d53baf38c148e50cc1f5a6a6f285d68862,0xcf6a714618a328c608a1c70cb62a31a6bef3f9d0,BUY,3.09,9.09,6.00,6.00,2,9.09,1,3.09,6.00,9.09


In [39]:
df = (df.merge(mdf, on='condition_id', how='inner')
    .sort_values(['bucket', 'copyable_pnl_exposure'], ascending=[True, False])
    .reset_index(drop=True)
)
df['end_date_iso'] = pd.to_datetime(df['end_date_iso'], utc=True)
df = df[df['end_date_iso'] > split_dt].copy().reset_index(drop=True)

In [40]:
df_plot = df.groupby('end_date_iso').agg(
    scaled_copyable_pnl=('scaled_copyable_pnl', 'sum'),
    ending_exposure=('scaled_copyable_pnl_exposure', 'sum'),
).reset_index()

df.sort_values('bucket', inplace=True)
df['scaled_copyable_pnl_cumsum'] = df['scaled_copyable_pnl'].cumsum()

df_pnl = df[['bucket', 'scaled_copyable_pnl_cumsum']].copy()

# exposure ends at EOD of end_date, so shift by 24h
df_exposure = df_plot[['end_date_iso', 'ending_exposure']].copy()
df_exposure['end_date_iso'] = pd.to_datetime(df_exposure['end_date_iso']) + pd.Timedelta(days=1)
df_exposure.rename(columns={'end_date_iso': 'dt', 'ending_exposure': 'exposure_delta'}, inplace=True)

resolution_exposure = df_exposure.groupby('dt').agg(exposure_delta=('exposure_delta', 'sum')).reset_index()
resolution_exposure['exposure_delta'] = resolution_exposure['exposure_delta'] * -1

buy_exposure = df[['bucket', 'scaled_copyable_pnl_exposure']].rename(columns={'bucket': 'dt', 'scaled_copyable_pnl_exposure': 'exposure_delta'})

df_exposure = ( pd.concat([resolution_exposure, buy_exposure], ignore_index=True)
    .reset_index(drop=True)
)

df_exposure.sort_values('dt', inplace=True)
df_exposure['exposure'] = df_exposure['exposure_delta'].cumsum()

# df_plot['end_date_iso'] = pd.to_datetime(df_plot['end_date_iso']) + pd.Timedelta(days=1)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_exposure['dt'],
    y=df_exposure['exposure'],
    mode='lines',
    name='exposure'
))

fig.add_trace(go.Scatter(
    x=df_pnl['bucket'],
    y=df_pnl['scaled_copyable_pnl_cumsum'],
    mode='lines',
    name='scaled_copyable_pnl_cumsum'
))

fig.show()

In [41]:
import plotly.express as px

# Aggregate by bucket
bucket_pnl = df.groupby('bucket')['scaled_copyable_pnl'].sum().reset_index()
bucket_pnl['scaled_copyable_pnl'] = bucket_pnl['scaled_copyable_pnl'].cumsum()

# Plot with Plotly
fig = px.line(
    bucket_pnl,
    x='bucket',
    y='scaled_copyable_pnl',
    title='Total Scaled Copyable PnL by Time Bucket',
    labels={'bucket': 'Time Bucket', 'scaled_copyable_pnl': 'Scaled Copyable PnL'}
)
fig.show()

In [42]:
( 
    df.groupby('condition_id').agg(
        total_copyable_exposure=('copyable_pnl_exposure', 'sum'),
        total_copyable_qty=('copyable_qty', 'sum'),
        trade_pnl =('trade_pnl', 'sum'),
        copyable_pnl = ('copyable_pnl', 'sum'),
        trades=('condition_id', 'count'),
        wallets=('wallet', lambda x: x.nunique()),
 ).sort_values('copyable_pnl', ascending=False).head(10)
)

,total_copyable_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,wallets
condition_id,,,,,,
0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,298166.82,525553.05,354865.77,206881.88,173,20
0xab1209b73e1ce9ebb4905dad781496385b5102c048889502b174ed733de1506a,10492.65,37934.30,37848.38,20797.82,105,12
0x3094a2b925483a06aa72945a1472e311e5eb6be75284f61e0c008e279508ddf6,19455.08,42527.80,12745.53,7985.26,47,10
0x46dd571d8c32bd58dcc553ae6c2a1ae8dda6d1ec69943bc0f771ea5c35132ca5,5955.13,13223.79,3638.41,2559.24,36,8
0x2de8d7b9d103366e37f3a448db8289c5c571dec5c0b9eeddbc673546d4bd9322,3173.36,10567.41,3908.47,2483.98,43,5
0xa6e388d87a81a17dd8f2046479e11c50bf32bcf475ce75a3506c5cf19a09613c,2175.47,5407.48,7099.32,2054.52,37,10
0xed25d03a2589af03a7a603e7c07c36d53baf38c148e50cc1f5a6a6f285d68862,1159.81,3337.55,8031.55,1927.51,19,6
0xfa08848205da5717b79e5bc21e7dbd31264fb01acc98bd1436b95870e8fec409,5685.41,10315.15,6639.67,1525.16,32,8
0x4815b9206b324492a71047496aadea8eb20e39b363792a6f8e828643361f5aa6,3334.12,5331.57,2986.86,1464.16,38,6


In [43]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

else:
    print("PLOT_WALLET_PNL=False — skipping plots")